# Simulating segmentation-quality inconsistency across BraTS sites

Real federated BraTS data doesn't come from equally careful annotators: some
sites over-segment (looser, more inclusive contours), some under-segment
(tighter, conservative contours). This notebook uses
`fed_brats_nnunet.augmentation.mask_perturbation.MorphologicalMaskPerturber`
to synthesize that kind of inconsistency by growing (dilating) or shrinking
(eroding) each tumor sub-region of a segmentation mask by a target volume
change (5-10% by default).

The perturbation logic itself lives in the package (`src/fed_brats_nnunet/augmentation/mask_perturbation.py`),
not in this notebook — this notebook is just a runnable demo of that reusable
class. Re-run it on any case, any dataset, any volume-change range.

In [ ]:
import sys
from pathlib import Path

# Find the repo root (the first parent that has a src/ folder) and put it on sys.path,
# so `import fed_brats_nnunet...` works regardless of where Jupyter was launched from.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from fed_brats_nnunet.augmentation.mask_perturbation import MorphologicalMaskPerturber

## 1. Quick self-contained demo

This cell needs no BraTS data at all — it builds a synthetic nested-sphere
segmentation (background / ED / NCR / ET, same label layout as a BraTS case)
so you can see the API and the visualization working immediately.

In [ ]:
def make_synthetic_segmentation(shape=(80, 80, 80)):
    """A nested-sphere mask standing in for one BraTS case's segmentation."""
    zz, yy, xx = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    center = np.array(shape) / 2
    distance = np.sqrt((zz - center[0]) ** 2 + (yy - center[1]) ** 2 + (xx - center[2]) ** 2)

    segmentation = np.zeros(shape, dtype=np.uint8)
    segmentation[distance < 24] = 1  # ED  (edema)
    segmentation[distance < 13] = 2  # NCR (necrotic core)
    segmentation[distance < 6] = 3   # ET  (enhancing tumor)
    return segmentation


segmentation = make_synthetic_segmentation()

perturber = MorphologicalMaskPerturber(volume_change_range=(0.05, 0.10), seed=42)
perturbed, reports = perturber.perturb_label_map(segmentation)

for report in reports:
    print(
        f"label {report.label}: {report.operation:>6} | "
        f"{report.original_voxels:>6} -> {report.perturbed_voxels:>6} voxels "
        f"({report.volume_change:+.1%})"
    )

In [ ]:
def show_before_after(before, after, slice_axis=0, slice_index=None, title=""):
    """Central-slice comparison: original mask, perturbed mask, voxels that differ."""
    if slice_index is None:
        slice_index = before.shape[slice_axis] // 2

    before_slice = np.take(before, slice_index, axis=slice_axis)
    after_slice = np.take(after, slice_index, axis=slice_axis)
    diff_slice = (before_slice != after_slice).astype(int)

    cmap = plt.get_cmap("tab10")
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

    axes[0].imshow(np.ma.masked_where(before_slice == 0, before_slice), cmap=cmap, vmin=0, vmax=9)
    axes[0].set_title("Original")

    axes[1].imshow(np.ma.masked_where(after_slice == 0, after_slice), cmap=cmap, vmin=0, vmax=9)
    axes[1].set_title("Perturbed")

    axes[2].imshow(diff_slice, cmap="Reds", vmin=0, vmax=1)
    axes[2].set_title("Changed voxels")

    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{title} (slice {slice_index} of axis {slice_axis})")
    fig.tight_layout()
    plt.show()


show_before_after(segmentation, perturbed, title="Synthetic case: erosion/dilation noise")

## 2. Running it on a real BraTS case

`BraTSDataset` (from the earlier dataset-conversion refactor) resolves a case
ID to its file paths, so this works whether you point it at your raw BraTS21
data (`_seg.nii.gz`, labels `{0,1,2,4}`) or at an already-converted nnUNet
`labelsTr` file (labels `{0,1,2,3}`) — the perturber is label-value agnostic,
it just discovers whichever non-zero labels are present.

Edit `DATASET_DIR` / `CASE_ID` below to point at a real case.

In [ ]:
from fed_brats_nnunet.data.case import BraTSDataset

DATASET_DIR = PROJECT_ROOT / "raw" / "dataset"  # adjust if your raw data lives elsewhere
CASE_ID = "BraTS2021_00000"  # replace with a real case id

dataset = BraTSDataset(DATASET_DIR)
case = dataset.case(CASE_ID)

import SimpleITK as sitk

label_image = sitk.ReadImage(str(case.label_path))
label_array = sitk.GetArrayFromImage(label_image)

case_perturber = MorphologicalMaskPerturber(volume_change_range=(0.05, 0.10), seed=0)
perturbed_array, case_reports = case_perturber.perturb_label_map(label_array)

for report in case_reports:
    print(f"label {report.label}: {report.operation:>6} | {report.volume_change:+.1%}")

show_before_after(label_array, perturbed_array, title=CASE_ID)

In [ ]:
# Write the perturbed segmentation back out as a .nii.gz, preserving the
# original image's spacing/origin/direction (perturb_file does the I/O for you).
OUTPUT_DIR = PROJECT_ROOT / "raw" / "noisy_dataset" / CASE_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

case_perturber.perturb_file(
    case.label_path,
    OUTPUT_DIR / f"{CASE_ID}_seg.nii.gz",
)

## 3. Batch: synthesizing a whole "noisy" site

Point this at a site's `labelsTr` folder (post nnUNet-conversion) or a raw
`labelsTr`-equivalent folder of `*_seg.nii.gz` files, and it perturbs every
case in parallel. Each file gets its own derived random seed so the batch is
reproducible but not identical across files.

In [ ]:
LABELS_TR_DIR = PROJECT_ROOT / "nnUNet_raw" / "Dataset137_BraTS2021_site_01" / "labelsTr"
NOISY_LABELS_DIR = PROJECT_ROOT / "nnUNet_raw" / "Dataset137_BraTS2021_site_01" / "labelsTr_noisy"

batch_perturber = MorphologicalMaskPerturber(volume_change_range=(0.05, 0.10), seed=123)
batch_reports = batch_perturber.perturb_folder(
    LABELS_TR_DIR, NOISY_LABELS_DIR, num_processes=4
)

import pandas as pd

rows = [
    {"file": filename, "label": report.label, "operation": report.operation, "volume_change": report.volume_change}
    for filename, reports in batch_reports.items()
    for report in reports
]
summary = pd.DataFrame(rows)
summary.groupby("label")["volume_change"].describe()[["mean", "std", "min", "max"]]

## Notes

- **`volume_change_range`** — `(low, high)` fraction of a label's original
  voxel count to add/remove. Each label gets its own random target drawn
  from this range, so a case's ED, NCR, and ET don't all shift by the same
  amount.
- **`operation`** — `"random"` (default, coin-flip per label), or force
  `"dilate"` / `"erode"` for every label.
- **`labels`** — restrict perturbation to specific label values; `None`
  (default) perturbs every non-zero label found in the array.
- **Nested-label priority** — labels are repainted largest-volume-first, so
  a smaller structure nested inside a bigger one (ET inside NCR inside ED)
  always survives on top of the bigger label's dilation.
- **`seed`** — pass an `int` for reproducible perturbations; omit it for a
  fresh random draw each run. `perturb_folder` derives a distinct seed per
  file from the perturber's own seed, so results are reproducible per-batch
  too.